### Getting Started:
- Make sure you are outside the sam2 repository, then run `pip install -r requirements.txt`

In [1]:
import torch
import numpy as np
from PIL import Image
from sam2.build_sam import build_sam2
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator
import matplotlib.pyplot as plt
import cv2

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"using device: {device}")

if device.type == "cuda":
    # use bfloat16 for the entire notebook
    torch.autocast("cuda", dtype=torch.bfloat16).__enter__()
    # turn on tfloat32 for Ampere GPUs (https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices)
    if torch.cuda.get_device_properties(0).major >= 8:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
elif device.type == "mps":
    print(
        "\nSupport for MPS devices is preliminary. SAM 2 is trained with CUDA and might "
        "give numerically different outputs and sometimes degraded performance on MPS. "
        "See e.g. https://github.com/pytorch/pytorch/issues/84936 for a discussion."
    )

In [3]:
def show_anns(anns, borders=True, path='example_image.png'):
    """
    NOTE: This function assumes you are saving to a folder called 'output_masks' 
    in the parent dir.
    """
    if len(anns) == 0:
        return
    sorted_anns = sorted(anns, key=(lambda x: x['area']), reverse=True)
    ax = plt.gca()
    ax.set_autoscale_on(False)
 
    img = np.ones((sorted_anns[0]['segmentation'].shape[0], sorted_anns[0]['segmentation'].shape[1], 4))
    # print("Initial canvas:", np.array(img))
    
    # Set default pixel transparency to ... (0 for transparent, 1 for full)
    img[:,:,3] = 1 

    # At this point, sorted_anns is a list of the segmented masks SAM2 found in the data
    for ann in sorted_anns:
        # For every area in the segmentation, get the mask at anns['segmentation']...
        m = ann['segmentation']
        
        # Use a color (that is NOT white)
        color_mask = np.concatenate([np.random.uniform(low=0.1, high=1, size=(3)), [1]])
        img[m] = color_mask 
        # and plot the colors
        if borders:
            contours, _ = cv2.findContours(m.astype(np.uint8),cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE) 
            # Try to smooth contours
            contours = [cv2.approxPolyDP(contour, epsilon=0.01, closed=True) for contour in contours]
            cv2.drawContours(img, contours, -1, (0,0,1,0.4), thickness=1) 
 

    ax.imshow(img)
    # print("Anns mask", sorted_anns[0]['segmentation'].shape)
    # print("Image as array:", np.array(img))

    

    plt.imsave(f"../output_masks/{path}", img)

In [4]:
def noisify_image(image:np.array, noise:str="random", threshold=0.1):
    # add noise to image
    # snow is generally greyscale, so noise values should be in that range
        # uint8 dtype limits cv2 randn to [0, 255]
    noise_mask = np.zeros(shape=(image.shape[0], image.shape[1], 3), dtype=np.uint8)

    if noise == "gaussian":
        # apply gaussian noise
        cv2.randn(noise_mask, mean=(128, 128, 128), stddev=(40, 40, 40))
        noise_mask = (noise_mask * 0.5).astype(np.uint8) # dilute the noise so that its application to the iamge is more realistic
        image = np.add(image, noise_mask)

    # add more noises methods here...
    elif noise == "random":
        for i in range(image.shape[0]):
            for j in range(image.shape[1]):
                if np.random.random() <= threshold:
                    image[i][j] = (np.random.rand(3) * 255).astype(np.uint32)
        

    return image

In [5]:
checkpoint = "./checkpoints/sam2.1_hiera_large.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_l.yaml"
mask_generator = SAM2AutomaticMaskGenerator(build_sam2(model_cfg, checkpoint, device=device, apply_postprocessing=False))

In [ ]:
image=Image.open("../0001TP_009240.png")
image=np.array(image.convert("RGB"))

print("Before Noise:")
plt.figure(figsize=(20, 20))
plt.imshow(image)
plt.axis('off')
plt.show()

# thresholds 0.3 and above are a bit brutal - like a snow day snowstorm.
image = noisify_image(image, noise="random", threshold=0.1)

print("After Noise:")
plt.figure(figsize=(20, 20))
plt.imshow(image)
plt.axis('off')
plt.show()

In [ ]:
masks = mask_generator.generate(image)

plt.figure(figsize=(20,20))
plt.imshow(image)
show_anns(masks)
plt.axis('off')
plt.show() 

## Common Bugs:
- "Torch is not compiled with Cuda" - uninstall torch, torchvision, torchaudio (`pip uninstall torch torchvision torchaudio`) and install the newest versions of each w/ Cuda (command available on Pytorch website). 
    - ATM this is:
     ```pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124```
- "Not implemented error" - this issue occurs when the device is Cuda but the Cuda version is not compatible with the GPU. Again, same fix as above (essentially update your Cuda version)
- "Commit denied by prereceive hook" - this issue occurs when the notebook is run & committed with all output. The output images in the notebook cause it to grow beyond git's acceptable size (100-120 mb). To fix, clear all output and restart kernel before committing.

In [8]:
import os
import random

dataset_path = "../download_dataset"
output_path = "../output_masks/val"

val_dir = os.path.join(dataset_path, 'val')

In [ ]:
k = 10
val_files = os.listdir(val_dir)
noise="random" # Disable by setting to None
show_img = True

for file in random.choices(val_files, k=k):
    try:
        src = os.path.join(val_dir, file)
        
        # Get original image
        image=Image.open(src)
        image=np.array(image.convert("RGB"))
        
        if show_img:
            plt.figure(figsize=(20, 20))
            plt.imshow(image)
            plt.axis('off')
            plt.show()
            print("Original Image")

        if noise:
            image = noisify_image(image)
            
            if show_img:
                print("After Noise:")
                plt.figure(figsize=(20, 20))
                plt.imshow(image)
                plt.axis('off')
                plt.show()

        # Show SAM2 Output
        masks = mask_generator.generate(image)
        plt.figure(figsize=(20,20))
        plt.imshow(image)
        show_anns(masks, path=f"val/{file}")
        plt.axis('off')
        plt.show() 
        print("SAM2 Segmented Output")

        # Show Ground Truth
        file_labelled = file[:-4] + "_L" + file[-4:] # labelled files have a '_L' in them
        src_labelled = os.path.join(dataset_path, 'val_labels')
        src_labelled = os.path.join(src_labelled, file_labelled) # Get the related labeled image filepath
        image=Image.open(src_labelled)
        image=np.array(image.convert("RGB"))
        plt.figure(figsize=(20, 20))
        plt.imshow(image)
        plt.axis('off')
        plt.show()
        print("Ground Truth")
    except:
        print("An image could not be found. Moving on...")